# 第7回　推定：点推定と区間推定
## ―― 「信頼区間95%」の、本当の意味

統計学Ⅰ（B）　／　北星学園大学

第6回で「標本平均は母平均の周りに SE=σ/√n で散らばる」と分かった。今日はそれを使って母平均を**区間**で推定する。注目は ――

> 「95%信頼区間」の95%は、**あなたが思っている意味とは、たぶん違う。**

### まず、自分の答えを書いてみよう

> **「95%信頼区間」とはどういう意味だと思いますか？**
>
> 多くの人はこう答える：「真の値が、95%の確率でこの区間の中にある」。
>
> ――この答え、実は**誤り**だ。なぜかを、今日シミュレーションで確かめる。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan

母集団 = df.loc[df["世帯年収万円"] < 5000, "世帯年収万円"].values
mu = 母集団.mean()        # 母平均（神様だけが知る真の値。今回はシミュレーションなので我々も知っている）
sigma = 母集団.std()      # 母標準偏差
n = 30
SE = sigma / np.sqrt(n)
print(f"母平均 μ = {mu:.0f}（本来は分からない）　母σ = {sigma:.0f}　n = {n}　SE = {SE:.1f}")

---
## 1. 点推定 ―― 1つの値で当てる

標本平均を、母平均の**推定値**として使う。これが**点推定**。でも標本ごとに値は揺れ、母平均にぴったりは当たらない。

In [ ]:
rng = np.random.default_rng(2026)
for i in range(5):
    xbar = rng.choice(母集団, n).mean()
    print(f"標本{i+1} の点推定（標本平均）= {xbar:.0f}　（母平均 {mu:.0f} とのズレ {xbar-mu:+.0f}）")
print("\n点推定は『当たり』を言い切るが、必ず少しズレる。どれくらい外れうるか＝幅が欲しい。")

---
## 2. 区間推定 ―― 幅を持たせる

そこで「だいたいこの範囲」と幅で示す。標本平均の散らばりは SE。正規分布では95%が平均±1.96SDに入るので、

$$ 95\%信頼区間 = \bar{x} \pm 1.96 \times SE $$

In [ ]:
rng = np.random.default_rng(2026)
xbar = rng.choice(母集団, n).mean()
half = 1.96 * SE
print(f"ある標本の平均 x̄ = {xbar:.0f}")
print(f"95%信頼区間 = [{xbar-half:.0f}, {xbar+half:.0f}]（半幅 {half:.0f}）")
print(f"→ この区間は母平均 {mu:.0f} を含んでいる")

---
## 3. では「95%」とは何の確率か？

ここが核心。多くの人は「**真の母平均が、95%の確率でこの区間に入る**」と思う。だが ――

**母平均 μ は固定された定数**だ（619で動かない）。区間 `[478, 682]` も計算してしまえば固定。固定した値が固定した区間に「95%の確率で入る」は意味をなさない。入っているか・いないか、どちらかでしかない。

では95%は何か。**ランダムなのは『区間のほう』**だ。標本を取り直すたびに区間は動く。**その手続きを何度も繰り返せば、作った区間の約95%が母平均を含む。** ―― これが95%の正体。確かめよう。

In [ ]:
rng = np.random.default_rng(2026)
plt.figure(figsize=(7, 8))
含んだ = 0
for i in range(100):
    xbar = rng.choice(母集団, n).mean()
    lo, hi = xbar - 1.96*SE, xbar + 1.96*SE
    当たり = lo <= mu <= hi
    含んだ += 当たり
    plt.plot([lo, hi], [i, i], color="#00897b" if 当たり else "#e8503a",
             lw=1.5, alpha=0.8)
plt.axvline(mu, color="#1565c0", lw=2, label=f"母平均 μ={mu:.0f}（真の値・固定）")
plt.xlabel("世帯年収（万円）"); plt.ylabel("標本の番号（1〜100）")
plt.title(f"100本の95%信頼区間：母平均を含んだのは {含んだ} 本（赤＝外した）")
plt.legend(loc="lower right"); plt.show()
print(f"100本中 {含んだ} 本が母平均を含んだ → これが『95%』の意味")

**100本中およそ95本**が母平均（青線・固定）を含み、約5本（赤）は外した。

「95%」とは、**この区間作りを繰り返したときの“当たり率”**であって、「いま手元の1本に真値が95%で入っている」ではない。手元の1本は、当たっているか外しているかのどちらかだ（どちらかは分からないが）。

---
## 4. 信頼度と幅のトレードオフ

「もっと確実に当てたい」なら信頼度を上げる（95%→99%）。すると**区間は広がる**。確実さと、区間の狭さ（情報の精密さ）は両立しない。

In [ ]:
for 信頼度, z in [("90%", 1.645), ("95%", 1.96), ("99%", 2.576)]:
    print(f"{信頼度}信頼区間の半幅 = {z}×SE = {z*SE:.0f} 万円")
print("\n確実にしたい（99%）ほど区間は広く＝ぼんやりする。")
print("極端に『100%確実』にすると幅は無限大（『年収は0〜∞万円』）＝何も言っていないのと同じ。")

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 点推定 | 標本平均1つで母平均を当てる。必ず少しズレる |
| 区間推定 | x̄ ± 1.96×SE で「だいたいこの範囲」 |
| 95%の意味 | 手続きを繰り返すと**約95%の区間が真値を含む**（当たり率） |
| よくある誤り | 「真値が95%でこの区間に入る」「μが確率的に動く」 |
| 信頼度と幅 | 確実にするほど区間は広がる（トレードオフ） |

> **母平均は固定。ランダムなのは区間のほう。**
> 「95%」は、この区間作りを100回やれば約95回当たる、という手続きの当たり率だ。

次回からは「差があるか」を判定する**仮説検定**へ。

**課題（Moodle）**：「95%信頼区間」のよくある誤った説明文の、どこが誤りかを正す。